In [25]:
include("GenX.jl")
#using .GenX
using HiGHS
using JuMP
using Gurobi
using DataFrames

┌ Info: Running precompile script for GenX. This may take a few minutes.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\startup\genx_startup.jl:56


  ____           __  __   _ _
 / ___| ___ _ __ \ \/ /  (_) |
| |  _ / _ \ '_ \ \  /   | | |
| |_| |  __/ | | |/  \ _ | | |
 \____|\___|_| |_/_/\_(_)/ |_|
                       |__/
 Version: nothing


In [26]:

# Genx case_runners file
case = "..\\example_systems\\15_markets_prm_ro_test"
settings_path = joinpath(case, "settings")
policies_path = joinpath(case, "policies")
output_folder = joinpath(case, "Results") # Write-output settings YAML file path
genx_settings = joinpath(settings_path, "genx_settings.yml") # Settings YAML file path


mysetup = GenX.configure_settings(genx_settings, output_folder) # mysetup dictionary stores settings and GenX-specific parameters

optimizer = Gurobi.Optimizer
OPTIMIZER =  GenX.configure_solver(settings_path, optimizer)
myinputs =  GenX.load_inputs(mysetup, case)



Configuring Settings
Reading Input CSV Files
Network.csv Successfully Read!
inputs[REP_PERIOD]1
inputs[H]8760
Demand (load) data Successfully Read!
Fuels_data.csv Successfully Read!


┌ Info: Thermal.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Vre.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Storage.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Must_run.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Vre_stor.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:336
┌ Info: Resource_energy_share_requirement.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:727
┌ Info: Resource_capacity_reserve_margin.csv Successfully Read.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\load_inputs\load_resources_data.jl:727
┌ Info: Resource_maximum_capacity_require


Summary of resources loaded into the model:
-------------------------------------------------------
	Resource type 		Number of resources
	Thermal        		13
	VRE            		11
	Storage        		16
	Must_run       		7
	VRE_and_storage		4
Total number of resources: 51
-------------------------------------------------------
Generators_variability.csv Successfully Read!
Validating time basis
Maximum_capacity_requirement.csv Successfully Read!
Energy_share_requirement.csv Successfully Read!
CO2_cap.csv Successfully Read!
Planning_reserve_margin.csv Successfully Read!
..\example_systems\15_markets_prm_ro_test\system\Vre_and_stor_solar_variability.csv Successfully Read!
Zones_markets.csv Successfully Read!
Market_price.csv Successfully Read!
Configuring RO Settings
Market_buy_price_bounds.csv Successfully Read!
Market_sell_price_bounds.csv Successfully Read!
CSV Files Successfully Read In From ..\example_systems\15_markets_prm_ro_test


Dict{Any, Any} with 135 entries:
  "Z"                           => 2
  "VS_ELEC"                     => Int64[]
  "RETROFIT_CAP"                => Int64[]
  "VS_STOR_AC_CHARGE"           => [48, 49, 50, 51]
  "LOSS_LINES"                  => Int64[]
  "dfMaxCO2"                    => [2.0e11;;]
  "STOR_HYDRO_SHORT_DURATION"   => Int64[]
  "RET_CAP_CHARGE"              => Set{Int64}()
  "pC_D_Curtail"                => [5000.0]
  "VS_STOR_DC_DISCHARGE"        => Int64[]
  "NEW_CAP_DC"                  => Int64[]
  "RESOURCE_NAMES_DC_DISCHARGE" => Any[]
  "ZONES_ELEC"                  => Any[]
  "Market_BuyPrices_Delta"      => [0.0 0.0 … 0.0 0.0; 12.4 17.4 … 15.0 26.6]
  "pTrans_Max_Possible"         => [200.0]
  "pNet_Map"                    => [-1.0 1.0]
  "omega"                       => [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0…
  "RET_CAP_ENERGY"              => Int64[]
  "RET_CAP_STOR"                => Int64[]
  ⋮                             => ⋮

In [30]:
filename = "Market_buy_price_bounds.csv"
tools_path = joinpath(case, "tools")
bprice_df = GenX.load_dataframe(joinpath(tools_path, filename))
filename = "Market_buy_price_bounds.csv"

scale_factor = setup["ParameterScale"] == 1 ? ModelScalingFactor : 1

if nrow(bprice_df) != inputs["T"]
    @warn """Number of inputs for delta in market hourly prices doesn't match number of hours 
    in the system """ maxlog=1
end

In [35]:
inputs["count_uncertain_param"] = 0

0

In [37]:
delta_buy_price_mat = GenX.extract_matrix_from_dataframe(bprice_df, "BuyPriceDelta")
delta_buy_price_mat ./= 0.1
#insure that we have price value for every market
if size(delta_buy_price_mat, 2) != inputs["Z"]
    @warn """Hourly buy prices should be provided for every market""" maxlog=1
end

ro_markets = sum(all(iszero, delta_buy_price_mat[:, col]) for col in axes(delta_buy_price_mat, 2))
inputs["Market_BuyPrices_Delta"] = transpose(delta_buy_price_mat)
inputs["count_uncertain_param"] += ro_markets * inputs["T"]

17520

In [66]:
filename = "Fuels_data_bounds.csv"
df_fuels = GenX.load_dataframe(joinpath(tools_path, filename))

scale_factor = setup["ParameterScale"] == 1 ? ModelScalingFactor : 1

if nrow(df_fuels) != inputs["T"]
    @warn """Number of inputs for delta in market hourly fuel prices doesn't match number of hours 
    in the system """ maxlog=1
end

# Fuel delta costs for each fuel type
existing_fuels = names(df_fuels)[2:end]
for f in inputs["fuels"]
    if f ∉ existing_fuels
        df_fuels[!,f] .= 0
    end
end

fuel_delta_costs = Containers.DenseAxisArray(transpose(Matrix(df_fuels[1:end, 2:end])), inputs["fuels"],1:nrow(df_fuels))
fuel_delta_costs /= scale_factor
ro_fuels = sum(all(!iszero, fuel_delta_costs[f,:]) for f in axes(fuel_delta_costs, 1))

inputs["count_uncertain_param"] += ro_fuels * inputs["T"]
inputs["fuel_delta_costs"] = fuel_delta_costs

2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, ["Biomass", "CA_Natural_Gas", "Conventional_DR", "DefaultFuel", "Geothermal", "None"]
    Dimension 2, 1:8760
And data, a 6×8760 Matrix{Float64}:
 0.0   0.0  0.0   0.0   0.0   0.0   …  0.0   0.0   0.0   0.0   0.0  0.0
 8.75  7.8  8.47  5.52  6.93  2.06     2.94  7.86  6.59  8.52  4.5  7.31
 0.0   0.0  0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0  0.0
 0.0   0.0  0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0  0.0
 0.0   0.0  0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0  0.0
 0.0   0.0  0.0   0.0   0.0   0.0   …  0.0   0.0   0.0   0.0   0.0  0.0

In [68]:
inputs["fuel_delta_costs"]["CA_Natural_Gas",1]


8.75

In [ ]:
EP =  GenX.generate_model(mysetup, myinputs, OPTIMIZER)


EP, solve_time =  GenX.solve_model(EP, mysetup)
myinputs["solve_time"] = solve_time
inputs = myinputs
setup = mysetup

In [ ]:
sum(value.(EP[:vCHARGE_VRE_STOR]).data)

In [ ]:
T = myinputs["T"]

gen = myinputs["RESOURCES"]
STOR = myinputs["VS_STOR"]
@variable(EP, vCHARGE_VRE_STOR2[y in STOR, t = 1:T] >= 0)
# 2. Enable/disable grid-interfacing charge
for y in STOR, t in 1:T
    if GenX.can_chrg_from_grid(gen[y]) == 0
        println(t)
        fix(vCHARGE_VRE_STOR2[y,t], 0; force = true)
    end
end

In [ ]:
is_fixed(vCHARGE_VRE_STOR2[48,2])

In [ ]:
vCHARGE_VRE_STOR2

In [ ]:
fix(vCHARGE_VRE_STOR2[1,1], 0; force=true)


In [ ]:

fix(vCHARGE_VRE_STOR2[y,t], 0; force=true)




In [ ]:
## Cost results
gen = inputs["RESOURCES"]
SEG = inputs["SEG"]  # Number of lines
Z = inputs["Z"]     # Number of zones
T = inputs["T"]     # Number of time steps (hours)
VRE_STOR = inputs["VRE_STOR"]

cost_list = [
    "cTotal",
    "cFix",
    "cVar",
    "cFuel",
    "cNSE",
    "cStart",
    "cUnmetRsv",
    "cNetworkExp",
    "cUnmetPolicyPenalty",
    "cCO2"
]
if !isempty(VRE_STOR)
    push!(cost_list, "cGridConnection")
end

dfCost = DataFrame(Costs = cost_list)

cVar = value(EP[:eTotalCVarOut]) +
        (!isempty(inputs["STOR_ALL"]) ? value(EP[:eTotalCVarIn]) : 0.0) +
        (!isempty(inputs["FLEX"]) ? value(EP[:eTotalCVarFlexIn]) : 0.0)
cFix = value(EP[:eTotalCFix]) +
        (!isempty(inputs["STOR_ALL"]) ? value(EP[:eTotalCFixEnergy]) : 0.0) +
        (!isempty(inputs["STOR_ASYMMETRIC"]) ? value(EP[:eTotalCFixCharge]) : 0.0)

cFuel = value.(EP[:eTotalCFuelOut])

if !isempty(VRE_STOR)
    cFix += ((!isempty(inputs["VS_DC"]) ? value(EP[:eTotalCFixDC]) : 0.0) +
                (!isempty(inputs["VS_SOLAR"]) ? value(EP[:eTotalCFixSolar]) : 0.0) +
                (!isempty(inputs["VS_WIND"]) ? value(EP[:eTotalCFixWind]) : 0.0))
    cVar += ((!isempty(inputs["VS_SOLAR"]) ? value(EP[:eTotalCVarOutSolar]) : 0.0) +
                (!isempty(inputs["VS_WIND"]) ? value(EP[:eTotalCVarOutWind]) : 0.0))
    if !isempty(inputs["VS_STOR"])
        cFix += ((!isempty(inputs["VS_STOR"]) ? value(EP[:eTotalCFixStor]) : 0.0) +
                    (!isempty(inputs["VS_ASYM_DC_CHARGE"]) ?
                    value(EP[:eTotalCFixCharge_DC]) : 0.0) +
                    (!isempty(inputs["VS_ASYM_DC_DISCHARGE"]) ?
                    value(EP[:eTotalCFixDischarge_DC]) : 0.0) +
                    (!isempty(inputs["VS_ASYM_AC_CHARGE"]) ?
                    value(EP[:eTotalCFixCharge_AC]) : 0.0) +
                    (!isempty(inputs["VS_ASYM_AC_DISCHARGE"]) ?
                    value(EP[:eTotalCFixDischarge_AC]) : 0.0))
        cVar += (!isempty(inputs["VS_STOR"]) ? value(EP[:eTotalCVarStor]) : 0.0)
    end
    total_cost = [
        value(EP[:eObj]),
        cFix,
        cVar,
        cFuel,
        value(EP[:eTotalCNSE]),
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0
    ]
else
    total_cost = [
        value(EP[:eObj]),
        cFix,
        cVar,
        cFuel,
        value(EP[:eTotalCNSE]),
        0.0,
        0.0,
        0.0,
        0.0,
        0.0
    ]
end


In [6]:

dfCost[!, Symbol("Total")] = total_cost

if setup["ParameterScale"] == 1
    dfCost.Total *= ModelScalingFactor^2
end

if setup["UCommit"] >= 1
    dfCost[6, 2] = value(EP[:eTotalCStart]) + value(EP[:eTotalCFuelStart])
end

if setup["OperationalReserves"] == 1
    dfCost[7, 2] = value(EP[:eTotalCRsvPen])
end

if setup["NetworkExpansion"] == 1 && Z > 1
    dfCost[8, 2] = value(EP[:eTotalCNetworkExp])
end

if haskey(inputs, "dfCapRes_slack")
    dfCost[9, 2] += value(EP[:eCTotalCapResSlack])
end

if haskey(inputs, "dfESR_slack")
    dfCost[9, 2] += value(EP[:eCTotalESRSlack])
end

if haskey(inputs, "dfCO2Cap_slack")
    dfCost[9, 2] += value(EP[:eCTotalCO2CapSlack])
end

if haskey(inputs, "MinCapPriceCap")
    dfCost[9, 2] += value(EP[:eTotalCMinCapSlack])
end

if haskey(inputs, "H2DemandPriceCap")
    dfCost[9, 2] += value(EP[:eTotalCH2DemandSlack])
end

if !isempty(VRE_STOR)
    dfCost[!, 2][11] = value(EP[:eTotalCGrid]) *
                       (setup["ParameterScale"] == 1 ? ModelScalingFactor^2 : 1)
end

if any(GenX.co2_capture_fraction.(gen) .!= 0)
    dfCost[10, 2] += value(EP[:eTotaleCCO2Sequestration])
end

if setup["ParameterScale"] == 1
    dfCost[6, 2] *= ModelScalingFactor^2
    dfCost[7, 2] *= ModelScalingFactor^2
    dfCost[8, 2] *= ModelScalingFactor^2
    dfCost[9, 2] *= ModelScalingFactor^2
    dfCost[10, 2] *= ModelScalingFactor^2
end

In [ ]:
dfCost

In [ ]:
z=1
tempCTotal = 0.0
tempCFix = 0.0
tempCVar = 0.0
tempCFuel = 0.0
tempCStart = 0.0
tempCNSE = 0.0
tempHydrogenValue = 0.0
tempCCO2 = 0.0

Y_ZONE = GenX.resources_in_zone_by_rid(gen, z)
STOR_ALL_ZONE = intersect(inputs["STOR_ALL"], Y_ZONE)
STOR_ASYMMETRIC_ZONE = intersect(inputs["STOR_ASYMMETRIC"], Y_ZONE)
FLEX_ZONE = intersect(inputs["FLEX"], Y_ZONE)
COMMIT_ZONE = intersect(inputs["COMMIT"], Y_ZONE)
ELECTROLYZERS_ZONE = intersect(inputs["ELECTROLYZER"], Y_ZONE)
CCS_ZONE = intersect(inputs["CCS"], Y_ZONE)

eCFix = sum(value.(EP[:eCFix][Y_ZONE]))
tempCFix += eCFix
tempCTotal += eCFix

tempCVar = sum(value.(EP[:eCVar_out][Y_ZONE, :]))
tempCTotal += tempCVar

tempCFuel = sum(value.(EP[:ePlantCFuelOut][Y_ZONE, :]))
tempCTotal += tempCFuel

if !isempty(STOR_ALL_ZONE)
    eCVar_in = sum(value.(EP[:eCVar_in][STOR_ALL_ZONE, :]))
    tempCVar += eCVar_in
    eCFixEnergy = sum(value.(EP[:eCFixEnergy][STOR_ALL_ZONE]))
    tempCFix += eCFixEnergy
    tempCTotal += eCVar_in + eCFixEnergy
end
if !isempty(STOR_ASYMMETRIC_ZONE)
    eCFixCharge = sum(value.(EP[:eCFixCharge][STOR_ASYMMETRIC_ZONE]))
    tempCFix += eCFixCharge
    tempCTotal += eCFixCharge
end
if !isempty(FLEX_ZONE)
    eCVarFlex_in = sum(value.(EP[:eCVarFlex_in][FLEX_ZONE, :]))
    tempCVar += eCVarFlex_in
    tempCTotal += eCVarFlex_in
end
if !isempty(VRE_STOR)
    gen_VRE_STOR = gen.VreStorage
    Y_ZONE_VRE_STOR = GenX.resources_in_zone_by_rid(gen_VRE_STOR, z)

    # Fixed Costs
    eCFix_VRE_STOR = 0.0
    SOLAR_ZONE_VRE_STOR = intersect(Y_ZONE_VRE_STOR, inputs["VS_SOLAR"])
    if !isempty(SOLAR_ZONE_VRE_STOR)
        eCFix_VRE_STOR += sum(value.(EP[:eCFixSolar][SOLAR_ZONE_VRE_STOR]))
    end
    WIND_ZONE_VRE_STOR = intersect(Y_ZONE_VRE_STOR, inputs["VS_WIND"])
    if !isempty(WIND_ZONE_VRE_STOR)
        eCFix_VRE_STOR += sum(value.(EP[:eCFixWind][WIND_ZONE_VRE_STOR]))
    end

    DC_ZONE_VRE_STOR = intersect(Y_ZONE_VRE_STOR, inputs["VS_DC"])
    if !isempty(DC_ZONE_VRE_STOR)
        eCFix_VRE_STOR += sum(value.(EP[:eCFixDC][DC_ZONE_VRE_STOR]))
    end
    STOR_ALL_ZONE_VRE_STOR = intersect(inputs["VS_STOR"], Y_ZONE_VRE_STOR)
    if !isempty(STOR_ALL_ZONE_VRE_STOR)
        eCFix_VRE_STOR += sum(value.(EP[:eCFixEnergy_VS][STOR_ALL_ZONE_VRE_STOR]))
        DC_CHARGE_ALL_ZONE_VRE_STOR = intersect(inputs["VS_ASYM_DC_CHARGE"],
            Y_ZONE_VRE_STOR)
        if !isempty(DC_CHARGE_ALL_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixCharge_DC][DC_CHARGE_ALL_ZONE_VRE_STOR]))
        end
        DC_DISCHARGE_ALL_ZONE_VRE_STOR = intersect(inputs["VS_ASYM_DC_DISCHARGE"],
            Y_ZONE_VRE_STOR)
        if !isempty(DC_DISCHARGE_ALL_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixDischarge_DC][DC_DISCHARGE_ALL_ZONE_VRE_STOR]))
        end
        AC_DISCHARGE_ALL_ZONE_VRE_STOR = intersect(inputs["VS_ASYM_AC_DISCHARGE"],
            Y_ZONE_VRE_STOR)
        if !isempty(AC_DISCHARGE_ALL_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixDischarge_AC][AC_DISCHARGE_ALL_ZONE_VRE_STOR]))
        end
        AC_CHARGE_ALL_ZONE_VRE_STOR = intersect(inputs["VS_ASYM_AC_CHARGE"],
            Y_ZONE_VRE_STOR)
        if !isempty(AC_CHARGE_ALL_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixCharge_AC][AC_CHARGE_ALL_ZONE_VRE_STOR]))
        end
    end
    tempCFix += eCFix_VRE_STOR

    # Variable Costs
    eCVar_VRE_STOR = 0.0
    if !isempty(SOLAR_ZONE_VRE_STOR)
        eCVar_VRE_STOR += sum(value.(EP[:eCVarOutSolar][SOLAR_ZONE_VRE_STOR, :]))
    end
    if !isempty(WIND_ZONE_VRE_STOR)
        eCVar_VRE_STOR += sum(value.(EP[:eCVarOutWind][WIND_ZONE_VRE_STOR, :]))
    end
    if !isempty(STOR_ALL_ZONE_VRE_STOR)
        vom_map = Dict(DC_CHARGE_ALL_ZONE_VRE_STOR => :eCVar_Charge_DC,
            DC_DISCHARGE_ALL_ZONE_VRE_STOR => :eCVar_Discharge_DC,
            AC_DISCHARGE_ALL_ZONE_VRE_STOR => :eCVar_Discharge_AC,
            AC_CHARGE_ALL_ZONE_VRE_STOR => :eCVar_Charge_AC)
        for (set, symbol) in vom_map
            if !isempty(set)
                eCVar_VRE_STOR += sum(value.(EP[symbol][set, :]))
            end
        end
    end
    tempCVar += eCVar_VRE_STOR

    # Total Added Costs
    tempCTotal += (eCFix_VRE_STOR + eCVar_VRE_STOR)
end

if setup["UCommit"] >= 1 && !isempty(COMMIT_ZONE)
    eCStart = sum(value.(EP[:eCStart][COMMIT_ZONE, :])) +
              sum(value.(EP[:ePlantCFuelStart][COMMIT_ZONE, :]))
    tempCStart += eCStart
    tempCTotal += eCStart
end


tempCNSE = sum(value.(EP[:eCNSE][:, :, z]))
tempCTotal += tempCNSE

# if any(dfGen.CO2_Capture_Fraction .!=0)
if !isempty(CCS_ZONE)
    tempCCO2 = sum(value.(EP[:ePlantCCO2Sequestration][CCS_ZONE]))
    tempCTotal += tempCCO2
end

if setup["ParameterScale"] == 1
    tempCTotal *= ModelScalingFactor^2
    tempCFix *= ModelScalingFactor^2
    tempCVar *= ModelScalingFactor^2
    tempCFuel *= ModelScalingFactor^2
    tempCNSE *= ModelScalingFactor^2
    tempCStart *= ModelScalingFactor^2
    tempHydrogenValue *= ModelScalingFactor^2
    tempCCO2 *= ModelScalingFactor^2
end
temp_cost_list = [
    tempCTotal,
    tempCFix,
    tempCVar,
    tempCFuel,
    tempCNSE,
    tempCStart,
    "-",
    "-",
    "-",
    tempCCO2
]
if !isempty(VRE_STOR)
    push!(temp_cost_list, "-")
end


dfCost[!, Symbol("Zone$z")] = temp_cost_list

In [ ]:
z=2
tempCTotal = 0.0
tempCFix = 0.0
tempCVar = 0.0
tempCFuel = 0.0
tempCStart = 0.0
tempCNSE = 0.0
tempHydrogenValue = 0.0
tempCCO2 = 0.0

Y_ZONE = GenX.resources_in_zone_by_rid(gen, z)
STOR_ALL_ZONE = intersect(inputs["STOR_ALL"], Y_ZONE)
STOR_ASYMMETRIC_ZONE = intersect(inputs["STOR_ASYMMETRIC"], Y_ZONE)
FLEX_ZONE = intersect(inputs["FLEX"], Y_ZONE)
COMMIT_ZONE = intersect(inputs["COMMIT"], Y_ZONE)
ELECTROLYZERS_ZONE = intersect(inputs["ELECTROLYZER"], Y_ZONE)
CCS_ZONE = intersect(inputs["CCS"], Y_ZONE)

In [ ]:
Y_ZONE

In [ ]:
EP[:eCFix]

In [ ]:
EP[:eCFix][Y_ZONE]

In [ ]:
value.(EP[:eCFix][Y_ZONE])

In [ ]:
sum(value.(EP[:eCFix][Y_ZONE]))

In [ ]:
GenX.by_rid_res(1, :resource, inputs["RESOURCES"])

In [ ]:
EP[:eCFix]

In [20]:
open("cFix_values_old.txt", "w") do file
    for i in 1:51
        name = GenX.by_rid_res(i, :resource, inputs["RESOURCES"])
        cfix = EP[:eCFix][i]
        write(file, "$name => $cfix\n")
   end
end

In [ ]:
typeof(EP[:eCFix][12])

In [ ]:
Y_ZONE

In [ ]:
value.(EP[:eCFix][Y_ZONE])

In [ ]:
sum(value.(EP[:eCFix][Y_ZONE]))

In [ ]:
eCFix = sum(value.(EP[:eCFix][Y_ZONE]))

In [ ]:
eCFix = sum(value.(EP[:eCFix][Y_ZONE]))

In [ ]:
z=2
tempCTotal = 0.0
tempCFix = 0.0
tempCVar = 0.0
tempCFuel = 0.0
tempCStart = 0.0
tempCNSE = 0.0
tempHydrogenValue = 0.0
tempCCO2 = 0.0

Y_ZONE = GenX.resources_in_zone_by_rid(gen, z)
STOR_ALL_ZONE = intersect(inputs["STOR_ALL"], Y_ZONE)
STOR_ASYMMETRIC_ZONE = intersect(inputs["STOR_ASYMMETRIC"], Y_ZONE)
FLEX_ZONE = intersect(inputs["FLEX"], Y_ZONE)
COMMIT_ZONE = intersect(inputs["COMMIT"], Y_ZONE)
ELECTROLYZERS_ZONE = intersect(inputs["ELECTROLYZER"], Y_ZONE)
CCS_ZONE = intersect(inputs["CCS"], Y_ZONE)

eCFix = sum(value.(EP[:eCFix][Y_ZONE]))
tempCFix += eCFix
tempCTotal += eCFix

tempCVar = sum(value.(EP[:eCVar_out][Y_ZONE, :]))
tempCTotal += tempCVar

tempCFuel = sum(value.(EP[:ePlantCFuelOut][Y_ZONE, :]))
tempCTotal += tempCFuel

if !isempty(STOR_ALL_ZONE)
    eCVar_in = sum(value.(EP[:eCVar_in][STOR_ALL_ZONE, :]))
    tempCVar += eCVar_in
    eCFixEnergy = sum(value.(EP[:eCFixEnergy][STOR_ALL_ZONE]))
    tempCFix += eCFixEnergy
    tempCTotal += eCVar_in + eCFixEnergy
end
if !isempty(STOR_ASYMMETRIC_ZONE)
    eCFixCharge = sum(value.(EP[:eCFixCharge][STOR_ASYMMETRIC_ZONE]))
    tempCFix += eCFixCharge
    tempCTotal += eCFixCharge
end
if !isempty(FLEX_ZONE)
    eCVarFlex_in = sum(value.(EP[:eCVarFlex_in][FLEX_ZONE, :]))
    tempCVar += eCVarFlex_in
    tempCTotal += eCVarFlex_in
end
if !isempty(VRE_STOR)
    gen_VRE_STOR = gen.VreStorage
    Y_ZONE_VRE_STOR = GenX.resources_in_zone_by_rid(gen_VRE_STOR, z)

    # Fixed Costs
    eCFix_VRE_STOR = 0.0
    SOLAR_ZONE_VRE_STOR = intersect(Y_ZONE_VRE_STOR, inputs["VS_SOLAR"])
    if !isempty(SOLAR_ZONE_VRE_STOR)
        eCFix_VRE_STOR += sum(value.(EP[:eCFixSolar][SOLAR_ZONE_VRE_STOR]))
    end
    WIND_ZONE_VRE_STOR = intersect(Y_ZONE_VRE_STOR, inputs["VS_WIND"])
    if !isempty(WIND_ZONE_VRE_STOR)
        eCFix_VRE_STOR += sum(value.(EP[:eCFixWind][WIND_ZONE_VRE_STOR]))
    end
    ELEC_ZONE_VRE_STOR = intersect(Y_ZONE_VRE_STOR, inputs["VS_ELEC"])
    if !isempty(ELEC_ZONE_VRE_STOR)
        eCFix_VRE_STOR += sum(value.(EP[:eCFixElec][ELEC_ZONE_VRE_STOR]))
    end
    DC_ZONE_VRE_STOR = intersect(Y_ZONE_VRE_STOR, inputs["VS_DC"])
    if !isempty(DC_ZONE_VRE_STOR)
        eCFix_VRE_STOR += sum(value.(EP[:eCFixDC][DC_ZONE_VRE_STOR]))
    end
    STOR_ALL_ZONE_VRE_STOR = intersect(inputs["VS_STOR"], Y_ZONE_VRE_STOR)
    if !isempty(STOR_ALL_ZONE_VRE_STOR)
        eCFix_VRE_STOR += sum(value.(EP[:eCFixEnergy_VS][STOR_ALL_ZONE_VRE_STOR]))
        DC_CHARGE_ALL_ZONE_VRE_STOR = intersect(inputs["VS_ASYM_DC_CHARGE"],
            Y_ZONE_VRE_STOR)
        if !isempty(DC_CHARGE_ALL_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixCharge_DC][DC_CHARGE_ALL_ZONE_VRE_STOR]))
        end
        DC_DISCHARGE_ALL_ZONE_VRE_STOR = intersect(inputs["VS_ASYM_DC_DISCHARGE"],
            Y_ZONE_VRE_STOR)
        if !isempty(DC_DISCHARGE_ALL_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixDischarge_DC][DC_DISCHARGE_ALL_ZONE_VRE_STOR]))
        end
        AC_DISCHARGE_ALL_ZONE_VRE_STOR = intersect(inputs["VS_ASYM_AC_DISCHARGE"],
            Y_ZONE_VRE_STOR)
        if !isempty(AC_DISCHARGE_ALL_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixDischarge_AC][AC_DISCHARGE_ALL_ZONE_VRE_STOR]))
        end
        AC_CHARGE_ALL_ZONE_VRE_STOR = intersect(inputs["VS_ASYM_AC_CHARGE"],
            Y_ZONE_VRE_STOR)
        if !isempty(AC_CHARGE_ALL_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixCharge_AC][AC_CHARGE_ALL_ZONE_VRE_STOR]))
        end
    end
    tempCFix += eCFix_VRE_STOR

    # Variable Costs
    eCVar_VRE_STOR = 0.0
    if !isempty(SOLAR_ZONE_VRE_STOR)
        eCVar_VRE_STOR += sum(value.(EP[:eCVarOutSolar][SOLAR_ZONE_VRE_STOR, :]))
    end
    if !isempty(WIND_ZONE_VRE_STOR)
        eCVar_VRE_STOR += sum(value.(EP[:eCVarOutWind][WIND_ZONE_VRE_STOR, :]))
    end
    if !isempty(STOR_ALL_ZONE_VRE_STOR)
        vom_map = Dict(DC_CHARGE_ALL_ZONE_VRE_STOR => :eCVar_Charge_DC,
            DC_DISCHARGE_ALL_ZONE_VRE_STOR => :eCVar_Discharge_DC,
            AC_DISCHARGE_ALL_ZONE_VRE_STOR => :eCVar_Discharge_AC,
            AC_CHARGE_ALL_ZONE_VRE_STOR => :eCVar_Charge_AC)
        for (set, symbol) in vom_map
            if !isempty(set)
                eCVar_VRE_STOR += sum(value.(EP[symbol][set, :]))
            end
        end
    end
    tempCVar += eCVar_VRE_STOR

    # Total Added Costs
    tempCTotal += (eCFix_VRE_STOR + eCVar_VRE_STOR)
end

if setup["UCommit"] >= 1 && !isempty(COMMIT_ZONE)
    eCStart = sum(value.(EP[:eCStart][COMMIT_ZONE, :])) +
              sum(value.(EP[:ePlantCFuelStart][COMMIT_ZONE, :]))
    tempCStart += eCStart
    tempCTotal += eCStart
end

if !isempty(ELECTROLYZER_ALL) # both electrolyzers and VRE+storage with electrolyzer component
    tempHydrogenValue = 0.0
    if !isempty(ELECTROLYZERS_ZONE)
        tempHydrogenValue -= sum(value.(EP[:eHydrogenValue][ELECTROLYZERS_ZONE, :]))
    end
    if !isempty(VRE_STOR) && !isempty(ELEC_ZONE_VRE_STOR)
        tempHydrogenValue -= sum(value.(EP[:eHydrogenValue_vs][ELEC_ZONE_VRE_STOR, :]))
    end
    tempCTotal += tempHydrogenValue
end

tempCNSE = sum(value.(EP[:eCNSE][:, :, z]))
tempCTotal += tempCNSE

# if any(dfGen.CO2_Capture_Fraction .!=0)
if !isempty(CCS_ZONE)
    tempCCO2 = sum(value.(EP[:ePlantCCO2Sequestration][CCS_ZONE]))
    tempCTotal += tempCCO2
end

if setup["ParameterScale"] == 1
    tempCTotal *= ModelScalingFactor^2
    tempCFix *= ModelScalingFactor^2
    tempCVar *= ModelScalingFactor^2
    tempCFuel *= ModelScalingFactor^2
    tempCNSE *= ModelScalingFactor^2
    tempCStart *= ModelScalingFactor^2
    tempHydrogenValue *= ModelScalingFactor^2
    tempCCO2 *= ModelScalingFactor^2
end
temp_cost_list = [
    tempCTotal,
    tempCFix,
    tempCVar,
    tempCFuel,
    tempCNSE,
    tempCStart,
    "-",
    "-",
    "-",
    tempCCO2
]
if !isempty(VRE_STOR)
    push!(temp_cost_list, "-")
end
if !isempty(ELECTROLYZER_ALL)
    push!(temp_cost_list, tempHydrogenValue)
end

dfCost[!, Symbol("Zone$z")] = temp_cost_list

In [ ]:
for z in 1:Z
    tempCTotal = 0.0
    tempCFix = 0.0
    tempCVar = 0.0
    tempCFuel = 0.0
    tempCStart = 0.0
    tempCNSE = 0.0
    tempHydrogenValue = 0.0
    tempCCO2 = 0.0

    Y_ZONE = resources_in_zone_by_rid(gen, z)
    STOR_ALL_ZONE = intersect(inputs["STOR_ALL"], Y_ZONE)
    STOR_ASYMMETRIC_ZONE = intersect(inputs["STOR_ASYMMETRIC"], Y_ZONE)
    FLEX_ZONE = intersect(inputs["FLEX"], Y_ZONE)
    COMMIT_ZONE = intersect(inputs["COMMIT"], Y_ZONE)
    ELECTROLYZERS_ZONE = intersect(inputs["ELECTROLYZER"], Y_ZONE)
    CCS_ZONE = intersect(inputs["CCS"], Y_ZONE)

    eCFix = sum(value.(EP[:eCFix][Y_ZONE]))
    tempCFix += eCFix
    tempCTotal += eCFix

    tempCVar = sum(value.(EP[:eCVar_out][Y_ZONE, :]))
    tempCTotal += tempCVar

    tempCFuel = sum(value.(EP[:ePlantCFuelOut][Y_ZONE, :]))
    tempCTotal += tempCFuel

    if !isempty(STOR_ALL_ZONE)
        eCVar_in = sum(value.(EP[:eCVar_in][STOR_ALL_ZONE, :]))
        tempCVar += eCVar_in
        eCFixEnergy = sum(value.(EP[:eCFixEnergy][STOR_ALL_ZONE]))
        tempCFix += eCFixEnergy
        tempCTotal += eCVar_in + eCFixEnergy
    end
    if !isempty(STOR_ASYMMETRIC_ZONE)
        eCFixCharge = sum(value.(EP[:eCFixCharge][STOR_ASYMMETRIC_ZONE]))
        tempCFix += eCFixCharge
        tempCTotal += eCFixCharge
    end
    if !isempty(FLEX_ZONE)
        eCVarFlex_in = sum(value.(EP[:eCVarFlex_in][FLEX_ZONE, :]))
        tempCVar += eCVarFlex_in
        tempCTotal += eCVarFlex_in
    end
    if !isempty(VRE_STOR)
        gen_VRE_STOR = gen.VreStorage
        Y_ZONE_VRE_STOR = resources_in_zone_by_rid(gen_VRE_STOR, z)

        # Fixed Costs
        eCFix_VRE_STOR = 0.0
        SOLAR_ZONE_VRE_STOR = intersect(Y_ZONE_VRE_STOR, inputs["VS_SOLAR"])
        if !isempty(SOLAR_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixSolar][SOLAR_ZONE_VRE_STOR]))
        end
        WIND_ZONE_VRE_STOR = intersect(Y_ZONE_VRE_STOR, inputs["VS_WIND"])
        if !isempty(WIND_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixWind][WIND_ZONE_VRE_STOR]))
        end
        ELEC_ZONE_VRE_STOR = intersect(Y_ZONE_VRE_STOR, inputs["VS_ELEC"])
        if !isempty(ELEC_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixElec][ELEC_ZONE_VRE_STOR]))
        end
        DC_ZONE_VRE_STOR = intersect(Y_ZONE_VRE_STOR, inputs["VS_DC"])
        if !isempty(DC_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixDC][DC_ZONE_VRE_STOR]))
        end
        STOR_ALL_ZONE_VRE_STOR = intersect(inputs["VS_STOR"], Y_ZONE_VRE_STOR)
        if !isempty(STOR_ALL_ZONE_VRE_STOR)
            eCFix_VRE_STOR += sum(value.(EP[:eCFixEnergy_VS][STOR_ALL_ZONE_VRE_STOR]))
            DC_CHARGE_ALL_ZONE_VRE_STOR = intersect(inputs["VS_ASYM_DC_CHARGE"],
                Y_ZONE_VRE_STOR)
            if !isempty(DC_CHARGE_ALL_ZONE_VRE_STOR)
                eCFix_VRE_STOR += sum(value.(EP[:eCFixCharge_DC][DC_CHARGE_ALL_ZONE_VRE_STOR]))
            end
            DC_DISCHARGE_ALL_ZONE_VRE_STOR = intersect(inputs["VS_ASYM_DC_DISCHARGE"],
                Y_ZONE_VRE_STOR)
            if !isempty(DC_DISCHARGE_ALL_ZONE_VRE_STOR)
                eCFix_VRE_STOR += sum(value.(EP[:eCFixDischarge_DC][DC_DISCHARGE_ALL_ZONE_VRE_STOR]))
            end
            AC_DISCHARGE_ALL_ZONE_VRE_STOR = intersect(inputs["VS_ASYM_AC_DISCHARGE"],
                Y_ZONE_VRE_STOR)
            if !isempty(AC_DISCHARGE_ALL_ZONE_VRE_STOR)
                eCFix_VRE_STOR += sum(value.(EP[:eCFixDischarge_AC][AC_DISCHARGE_ALL_ZONE_VRE_STOR]))
            end
            AC_CHARGE_ALL_ZONE_VRE_STOR = intersect(inputs["VS_ASYM_AC_CHARGE"],
                Y_ZONE_VRE_STOR)
            if !isempty(AC_CHARGE_ALL_ZONE_VRE_STOR)
                eCFix_VRE_STOR += sum(value.(EP[:eCFixCharge_AC][AC_CHARGE_ALL_ZONE_VRE_STOR]))
            end
        end
        tempCFix += eCFix_VRE_STOR

        # Variable Costs
        eCVar_VRE_STOR = 0.0
        if !isempty(SOLAR_ZONE_VRE_STOR)
            eCVar_VRE_STOR += sum(value.(EP[:eCVarOutSolar][SOLAR_ZONE_VRE_STOR, :]))
        end
        if !isempty(WIND_ZONE_VRE_STOR)
            eCVar_VRE_STOR += sum(value.(EP[:eCVarOutWind][WIND_ZONE_VRE_STOR, :]))
        end
        if !isempty(STOR_ALL_ZONE_VRE_STOR)
            vom_map = Dict(DC_CHARGE_ALL_ZONE_VRE_STOR => :eCVar_Charge_DC,
                DC_DISCHARGE_ALL_ZONE_VRE_STOR => :eCVar_Discharge_DC,
                AC_DISCHARGE_ALL_ZONE_VRE_STOR => :eCVar_Discharge_AC,
                AC_CHARGE_ALL_ZONE_VRE_STOR => :eCVar_Charge_AC)
            for (set, symbol) in vom_map
                if !isempty(set)
                    eCVar_VRE_STOR += sum(value.(EP[symbol][set, :]))
                end
            end
        end
        tempCVar += eCVar_VRE_STOR

        # Total Added Costs
        tempCTotal += (eCFix_VRE_STOR + eCVar_VRE_STOR)
    end

    if setup["UCommit"] >= 1 && !isempty(COMMIT_ZONE)
        eCStart = sum(value.(EP[:eCStart][COMMIT_ZONE, :])) +
                  sum(value.(EP[:ePlantCFuelStart][COMMIT_ZONE, :]))
        tempCStart += eCStart
        tempCTotal += eCStart
    end

    if !isempty(ELECTROLYZER_ALL) # both electrolyzers and VRE+storage with electrolyzer component
        tempHydrogenValue = 0.0
        if !isempty(ELECTROLYZERS_ZONE)
            tempHydrogenValue -= sum(value.(EP[:eHydrogenValue][ELECTROLYZERS_ZONE, :]))
        end
        if !isempty(VRE_STOR) && !isempty(ELEC_ZONE_VRE_STOR)
            tempHydrogenValue -= sum(value.(EP[:eHydrogenValue_vs][ELEC_ZONE_VRE_STOR, :]))
        end
        tempCTotal += tempHydrogenValue
    end

    tempCNSE = sum(value.(EP[:eCNSE][:, :, z]))
    tempCTotal += tempCNSE

    # if any(dfGen.CO2_Capture_Fraction .!=0)
    if !isempty(CCS_ZONE)
        tempCCO2 = sum(value.(EP[:ePlantCCO2Sequestration][CCS_ZONE]))
        tempCTotal += tempCCO2
    end

    if setup["ParameterScale"] == 1
        tempCTotal *= ModelScalingFactor^2
        tempCFix *= ModelScalingFactor^2
        tempCVar *= ModelScalingFactor^2
        tempCFuel *= ModelScalingFactor^2
        tempCNSE *= ModelScalingFactor^2
        tempCStart *= ModelScalingFactor^2
        tempHydrogenValue *= ModelScalingFactor^2
        tempCCO2 *= ModelScalingFactor^2
    end
    temp_cost_list = [
        tempCTotal,
        tempCFix,
        tempCVar,
        tempCFuel,
        tempCNSE,
        tempCStart,
        "-",
        "-",
        "-",
        tempCCO2
    ]
    if !isempty(VRE_STOR)
        push!(temp_cost_list, "-")
    end
    if !isempty(ELECTROLYZER_ALL)
        push!(temp_cost_list, tempHydrogenValue)
    end

    dfCost[!, Symbol("Zone$z")] = temp_cost_list
end
CSV.write(joinpath(path, "costs.csv"), dfCost)
